In [1]:
%run "../0 - Base/inicializa.ipynb"

In [2]:
DF_Compras = pd.read_csv("..\\Raw\\RAW_COMPRAS_ENTRADAS.csv")

In [3]:
DF_Compras.head(5)

,NUM_NOTA_FISCAL,DATA_ENTRADA,FORNECEDOR,COD_PROD,QTD_COMPRADA,VALOR_TOTAL_NOTA,RESPONSAVEL_RECEBIMENTO
0,NF-1006,01/01/2026,CEASA SP - PRODUTORES UNIDOS,101,55,163.39,Mateus
1,NF-1046,01/01/2026,CEASA SP - PRODUTORES UNIDOS,102,119,335.78,Thiago
2,NF-1018,01/01/2026,CEASA SP - PRODUTORES UNIDOS,103,115 KG,243.92,Thiago
3,NF-1013,01/01/2026,CEASA SP - PRODUTORES UNIDOS,104,41,123.78,diego
4,NF-1004,01/01/2026,CEASA SP - PRODUTORES UNIDOS,105,85,235.48,Diego


## Trata NULL

In [4]:
DF_Compras['NUM_NOTA_FISCAL'].isna().value_counts()

NUM_NOTA_FISCAL
False    402
Name: count, dtype: int64

In [5]:
DF_Compras['FORNECEDOR'].isna().value_counts()

FORNECEDOR
False    402
Name: count, dtype: int64

In [6]:
DF_Compras['COD_PROD'].isna().value_counts()

COD_PROD
False    402
Name: count, dtype: int64

In [7]:
DF_Compras['QTD_COMPRADA'].isna().value_counts()

QTD_COMPRADA
False    402
Name: count, dtype: int64

In [8]:
DF_Compras['VALOR_TOTAL_NOTA'].isna().value_counts()

VALOR_TOTAL_NOTA
False    402
Name: count, dtype: int64

In [9]:
DF_Compras['RESPONSAVEL_RECEBIMENTO'].isna().value_counts()

RESPONSAVEL_RECEBIMENTO
False    355
True      47
Name: count, dtype: int64

-----

In [10]:
silverCompras = DF_Compras # Instancia tabela compras 

### Trata Cod_Prod

In [11]:
silverCompras['COD_PROD'] = silverCompras['COD_PROD'].fillna(1000001).replace(0, 1000001).astype('Int64') # Trata valores nulos com 1000001

### Trata NF

In [12]:
silverCompras['N_NF'] = silverCompras['NUM_NOTA_FISCAL'].str.replace('NF-','')

### Trata Fornecedor

In [13]:
silverCompras['FORNECEDOR'].value_counts()

FORNECEDOR
CEASA SP - PRODUTORES UNIDOS     242
FRIGORIFICO FRIBOI / BOI LEVE    160
Name: count, dtype: int64

In [14]:
silverCompras['FORNECEDOR'] = (
    silverCompras['FORNECEDOR']
    .str.replace(' - PRODUTORES UNIDOS','')
    .str.replace(' / BOI LEVE','')
)

In [15]:
silverCompras['FORNECEDOR'].value_counts()

FORNECEDOR
CEASA SP              242
FRIGORIFICO FRIBOI    160
Name: count, dtype: int64

### Trata Valor

In [16]:
# silverCompras['VALOR_TOTAL_NOTA'] = silverCompras['VALOR_TOTAL_NOTA'].str.replace(',','.')

In [17]:
# silverCompras['TOTAL_NOTA'] = pd.to_numeric(silverCompras['VALOR_TOTAL_NOTA'], errors="coerce")

limpo = (
    silverCompras['VALOR_TOTAL_NOTA'].astype(str)
    .str.replace('R$', '', regex=False)
    .str.strip()
    .str.replace(',', '.', regex=False)
)
silverCompras['TOTAL_NOTA'] = pd.to_numeric(limpo, errors='coerce').fillna(0.0)

assert silverCompras['TOTAL_NOTA'].sum() > 222000, "total de compras abaixo do bruto"

In [18]:
silverCompras['TOTAL_NOTA'].isna().value_counts()

TOTAL_NOTA
False    402
Name: count, dtype: int64

In [19]:
  # silverCompras['TOTAL_NOTA'] = silverCompras['TOTAL_NOTA'].fillna(0).replace(0.0, 0).astype(int)

In [20]:
silverCompras['TOTAL_NOTA'].isna().value_counts()

TOTAL_NOTA
False    402
Name: count, dtype: int64

### Trata qtd

In [21]:
silverCompras['QTD_COMPRADA'] = silverCompras['QTD_COMPRADA'].str.replace(' KG','')

### Trata Resp

In [22]:
silverCompras['RESPONSAVEL_RECEBIMENTO'] = silverCompras['RESPONSAVEL_RECEBIMENTO'].str.upper()

### Trata Data

In [23]:
silverCompras['DATA'] = pd.to_datetime(silverCompras['DATA_ENTRADA'], format='%d/%m/%Y')

silverCompras['MES'] = silverCompras['DATA'].dt.month
silverCompras['ANO'] = silverCompras['DATA'].dt.year
silverCompras['DIA'] = silverCompras['DATA'].dt.day

In [24]:

DF_Compras.head(5)

,NUM_NOTA_FISCAL,DATA_ENTRADA,FORNECEDOR,COD_PROD,QTD_COMPRADA,VALOR_TOTAL_NOTA,RESPONSAVEL_RECEBIMENTO,N_NF,TOTAL_NOTA,DATA,MES,ANO,DIA
0,NF-1006,01/01/2026,CEASA SP,101,55,163.39,MATEUS,1006,163.39,2026-01-01,1,2026,1
1,NF-1046,01/01/2026,CEASA SP,102,119,335.78,THIAGO,1046,335.78,2026-01-01,1,2026,1
2,NF-1018,01/01/2026,CEASA SP,103,115,243.92,THIAGO,1018,243.92,2026-01-01,1,2026,1
3,NF-1013,01/01/2026,CEASA SP,104,41,123.78,DIEGO,1013,123.78,2026-01-01,1,2026,1
4,NF-1004,01/01/2026,CEASA SP,105,85,235.48,DIEGO,1004,235.48,2026-01-01,1,2026,1


------

In [25]:
resultado = False


### Cria codigo fornecedor

In [26]:
silverCompras['FORNECEDOR'].value_counts()

FORNECEDOR
CEASA SP              242
FRIGORIFICO FRIBOI    160
Name: count, dtype: int64

In [27]:
MAPA_FORNECEDOR = {
    'CEASA SP': 1001,
    'FRIGORIFICO FRIBOI': 1002,
}

silverCompras['ID_FORNECEDOR'] = (
    silverCompras['FORNECEDOR']
    .str.strip()
    .str.upper()
    .map(MAPA_FORNECEDOR)
    .fillna(0)
    .astype('Int64')
)

silverCompras['ID_FORNECEDOR'].head(5)

0    1001
1    1001
2    1001
3    1001
4    1001
Name: ID_FORNECEDOR, dtype: Int64

In [28]:
silverCompras['QTD_COMPRADA'] = silverCompras['QTD_COMPRADA'].astype(float)

silverCompras['PRECO_UNIT'] = silverCompras['TOTAL_NOTA'] / silverCompras['QTD_COMPRADA']

silverCompras['PRECO_UNIT'] = pd.to_numeric(silverCompras['PRECO_UNIT']).astype(float)
silverCompras['PRECO_UNIT'].value_counts()

PRECO_UNIT
2.82     2
35.89    2
2.97     2
17.17    2
14.01    2
        ..
3.18     1
18.82    1
21.61    1
21.81    1
25.46    1
Name: count, Length: 377, dtype: int64

In [29]:
# # silverCompras['DIA'] = silverCompras['DATA_ENTRADA'].split('/%d/%YYYY')
# # silverCompras['DIA'].head(5)

# silverCompras['DATA_ENTRADA'] = pd.to_datetime(
#     silverCompras['DATA_ENTRADA'],
#     format='%d/%m/%Y',
#     errors='coerce'
# )

# silverCompras['DIA'] = silverCompras['DATA_ENTRADA'].dt.day.astype('Int64')
# silverCompras['DIA'].head(5)

In [30]:
silverCompras.head(5)

,NUM_NOTA_FISCAL,DATA_ENTRADA,FORNECEDOR,COD_PROD,QTD_COMPRADA,VALOR_TOTAL_NOTA,RESPONSAVEL_RECEBIMENTO,N_NF,TOTAL_NOTA,DATA,MES,ANO,DIA,ID_FORNECEDOR,PRECO_UNIT
0,NF-1006,01/01/2026,CEASA SP,101,55.00,163.39,MATEUS,1006,163.39,2026-01-01,1,2026,1,1001,2.97
1,NF-1046,01/01/2026,CEASA SP,102,119.00,335.78,THIAGO,1046,335.78,2026-01-01,1,2026,1,1001,2.82
2,NF-1018,01/01/2026,CEASA SP,103,115.00,243.92,THIAGO,1018,243.92,2026-01-01,1,2026,1,1001,2.12
3,NF-1013,01/01/2026,CEASA SP,104,41.00,123.78,DIEGO,1013,123.78,2026-01-01,1,2026,1,1001,3.02
4,NF-1004,01/01/2026,CEASA SP,105,85.00,235.48,DIEGO,1004,235.48,2026-01-01,1,2026,1,1001,2.77


In [31]:
resultado = False

In [32]:
def export_dados_excel(resultado):

    pasta = Path("..\\Raw\\table_silver\\")
    pasta.mkdir(parents=True, exist_ok=True)

    if resultado:

        arquivo_excel = pasta / "silverCompras_ETL.xlsx"

        # 1. Criar os arquivos
        silverCompras.to_excel(arquivo_excel, index=False)

        # 3. Consolidar em um Excel com duas abas
        with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

            silverCompras.to_excel(
                writer,
                sheet_name="Excel",
                index=False
            )

        return True

    print("ERRO")
    return False

In [33]:
export_dados_excel(True)

True